In [ ]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
df=pd.read_csv("")
X=df.drop("target",axis=1)
y=df["target"]
# Separate numerical and categorical columns
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Numerical preprocessing pipeline
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numerical and categorical preprocessing
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# Complete pipeline: preprocessing -> PCA -> Logistic Regression
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("pca", PCA()),
    ("model", LogisticRegression(max_iter=2000))
])

# Hyperparameter grid
params = {
    "pca__n_components": [2, 3, 4],
    "model__C": [0.1, 1, 10]
}

# Grid search with 5-fold cross-validation
grid = GridSearchCV(
    estimator=pipe,
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train and tune the model
grid.fit(X_train, y_train)

# Predict on test data
pred = grid.best_estimator_.predict(X_test)

# Evaluate the model
print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)
print("Test Accuracy:", accuracy_score(y_test, pred))
print("\nClassification Report:\n")
print(classification_report(y_test, pred))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.stats import (
    ttest_rel, ttest_ind, wilcoxon,
    friedmanchisquare, f_oneway
)

outer_cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

scores_pca = cross_val_score(
    grid, X_train, y_train,
    cv=outer_cv, scoring="accuracy"
)

scores_no_pca = cross_val_score(
    grid_no_pca, X_train, y_train,
    cv=outer_cv, scoring="accuracy"
)

print("PCA scores:", scores_pca)
print("Without PCA scores:", scores_no_pca)

In [ ]:
from scipy.stats import shapiro

# Difference between paired CV scores
diff = scores_pca - scores_no_pca

stat, p = shapiro(diff)

if p > 0.05:
    print("Differences are approximately normal")
else:
    print("Differences may not be normally distributed")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import probplot

probplot(diff, dist="norm", plot=plt)
plt.title("Q-Q Plot of Paired Differences")
plt.show()

In [ ]:
#paired P
stat, p = ttest_rel(scores_pca, scores_no_pca)

if p < 0.05:
    print("Significant difference")
else:
    print("No significant difference detected")

In [ ]:
#wilcoxon
stat, p = wilcoxon(scores_pca, scores_no_pca)

if p < 0.05:
    print("Significant difference")
else:
    print("No significant difference detected")

In [ ]:
#friedman
stat, p = friedmanchisquare(
    scores_model1,
    scores_model2,
    scores_model3
)

if p < 0.05:
    print("Significant difference; consider a post-hoc test")
else:
    print("No significant difference detected")

In [ ]:
#oneway-annova
stat, p = f_oneway(
    scores_group1,
    scores_group2,
    scores_group3
)

if p < 0.05:
    print("At least one group mean differs significantly")
else:
    print("No significant difference detected")

In [ ]:
#independent t test
stat, p = ttest_ind(
    scores_group1,
    scores_group2,
    equal_var=False
)

if p < 0.05:
    print("Significant difference")
else:
    print("No significant difference detected")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow Method
inertia = []

for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

# Silhouette Score
silhouette_scores = []

for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis")
plt.show()

# Best k according to silhouette score
best_k = range(2, 11)[silhouette_scores.index(max(silhouette_scores))]
print("Best k:", best_k)
print("Best Silhouette Score:", max(silhouette_scores))

# Final K-Means model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print("Cluster Labels:", clusters)